# Graph Network
***

In this notebook I will explore how to create a graph of the station topology. I will find for each station in CAMELS-ES its inmediate downstream station, and then I will create a network (graph) of each river basin.

**To be improved**


- [ ] Add attributes to the station nodes: elevation, start and end, active...
- [ ] Add attributes to the dams: reservoir storage, reservoir area, maximum and minimum levels, dam height...
- [ ] Compute the degree of regulation of the stations based on all the reservoirs above.

In [ ]:
import pyflwdir
import numpy as np
import pandas as pd
import geopandas as gpd
import rioxarray as rxr
# from pathlib import Path
from tqdm.auto import tqdm
# import matplotlib.pyplot as plt
import networkx as nx
# import pyproj

from ocab.config import Config
from ocab.graph import *
from ocab.api import read_timeseries

## Config

In [ ]:
cfg_camels = Config('../CAMELS/config_CAMELS_v200.yml')
cfg_beaver = Config('../BEAVERS/config_BEAVERS_v100.yml')

## Data

### Digital Elevation Model

In [ ]:
# load flow direction map and convert it to `flwdir`
flwdir = rxr.open_rasterio(cfg_camels.path_merit / 'dir.tif').squeeze(dim='band')
crs = flwdir.rio.crs
flwdir = pyflwdir.from_array(
    flwdir.data,
    ftype='d8',
    transform=flwdir.rio.transform(),
    check_ftype=False,
    latlon=True
)

# Compute upstream area map in km²
uparea = flwdir.upstream_area(unit='km2')
mask_area = uparea >= 20

# Map of the Strahler order
strahler = flwdir.stream_order(type='strahler', mask=mask_area)

### Points
#### Gauging Stations

In [ ]:
stations = load_MERIT_points(
    path=cfg_camels.path_dataset / 'preprocessing' / 'basins' / 'output' / 'stations_outlets_3sec.geojson',
    flwdir=flwdir,
    crs=crs,
    kind='station'
)
print(f'{len(stations)} stations')

#### Dams

In [ ]:
dams = load_MERIT_points(
    path=cfg_beaver.path_dataset / 'preprocessing' / 'basins' / 'output' / 'dams_outlets_3sec.geojson',
    flwdir=flwdir,
    crs=crs,
    kind='dam'
)

print(f'{len(dams)} dams')

## Topological Processing

### Outlets

In this section I find the river mouth for every basin in the dataset.

In [ ]:
# find basin outlets
outlets = find_outlets(pd.concat([stations, dams], axis=0), flwdir, uparea)
print(f'{len(outlets)} outlets')

In [ ]:
outlets

> **Note**. It would be great to name each outlet after the basin codes: Duero 2000, Tajo 3000, Guadiana 4000, Guadalquivir 5000. The problem is that the IDs are not unique between stations and dams, that in some cases the x000 ID is already taken, and that small catchments in Cantábrico, Galicia Costa or Júcar are difficult to name.

### Find Downstream Point

In [ ]:
# merge all available points
points = pd.concat([stations, dams, outlets], axis=0).sort_index(axis=0)

# add or rename attributes
points['lat'] = points.geometry.y
points['lon'] = points.geometry.x
points.rename(
    columns={'area_skm': 'area', 'flwdir_index': 'pixel'}, 
    inplace=True, 
    errors='ignore'
    )
points['strahler'] = strahler.ravel()[points.pixel]

# find points downstream
pixel_to_id = dict(zip(points['pixel'], points.index))
ds_neighbours = [find_downstream_neighbour(pixel, flwdir, pixel_to_id) for pixel in points['pixel']]
neighbours, distances = zip(*ds_neighbours)
points['downstream_ID'] = neighbours
points['dist_to_outlet'] = distances

# compute distance to downstream station
dist_m = pd.Series(index=points.index, name='dist_m')
for ID in tqdm(dist_m.index):
    ds_ID = points.loc[ID, 'downstream_ID']
    if pd.isnull(ds_ID):
        dist_m[ID] = points.loc[ID, 'dist_to_outlet']
    else:
        dist_m[ID] = np.diff(points.loc[[ds_ID, ID], 'dist_to_outlet'])[0]
points['dist_m'] = dist_m

# # export
# # points.to_file(cfg_camels.path_gis / 'nodes.geojson', driver='GeoJSON')
# points.to_file('nodes.geojson', driver='GeoJSON')

***

In [ ]:
path_ts = Path('../../docs/timeseries')

for dam_ID in points.xs('dam', level='kind').index:
    print(dam_ID)
    break

In [ ]:
dam_ID = 2030

In [ ]:
point = points.loc[[(dam_ID, 'dam')]]

point

In [ ]:
# import reservoir timeseries
resops = read_timeseries(
    path_ts / 'reservoirs', 
    ID=dam_ID,
    variables=['storage_mcm', 'outflow_cms', 'inflow_cms']
)[dam_ID]
print(f'avg. inflow:\t\t{resops['inflow_cms'].mean():.1f} m³/s')
print(f'avg. outflow:\t\t{resops['outflow_cms'].mean():.1f} m³/s')

In [ ]:
# get upstream nodes
mask = points['downstream_ID'] == (dam_ID, 'dam')
upstream = points[mask]
print(f'No. upstream points:\t{mask.sum()}')

# contributed area covered by upstream nodes
area_up = (upstream['area'].sum() / point['area']).item()
print(f'area covered:\t\t{area_up * 100:.1f}%')

# read discharge upstream
discharge_up = read_timeseries(
    path_ts / 'stations',
    ID=upstream.index.get_level_values('id').tolist(),
    variables=['discharge_cms']
)
discharge_up = pd.concat(
    {ID: df['discharge_cms'] for ID, df in discharge_up.items()}, 
    axis=1
).sum(axis=1)
avg_dis_up = discharge_up.mean()
print(f'avg. inflow:\t\t{avg_dis_up:.1f} m³/s')
print(f'avg. inflow:\t\t{avg_dis_up / area_up:.1f} m³/s')

In [ ]:
# extract node downstream
downstream = points.loc[point['downstream_ID']]

# contributed area covered by upstream nodes
area_down = (downstream['area'].sum() / point['area']).item()
print(f'area covered:\t\t{area_down * 100:.1f}%')

# import discharge downstream
discharge_down = read_timeseries(
    path_ts / 'stations',
    ID=downstream.index.get_level_values('id').tolist(),
    variables=['discharge_cms']
)
discharge_down = pd.concat(
    {ID: df['discharge_cms'] for ID, df in discharge_down.items()},
    axis=1
).sum(axis=1)
avg_dis_down = discharge_down.mean()
print(f'avg. inflow:\t\t{avg_dis_down:.1f} m³/s')
print(f'avg. inflow:\t\t{avg_dis_down / area_down:.1f} m³/s')

In [ ]:
fig, ax = plt.subplots(nrows=2, figsize=(16, 6), sharex=True, sharey=True)
kwargs = {'lw': .8}

# INFLOW
resops['inflow_cms'].plot(ax=ax[0], label='est', **kwargs)
discharge_up.plot(ax=ax[0], label='obs', **kwargs)
(discharge_up / area_up).plot(ax=ax[0], label='obs_corr', **kwargs)
ax[0].set(
    ylabel='inflow (cms)'
);
ax[0].legend(frameon=False);

# OUTFLOW
resops['outflow_cms'].plot(ax=ax[1], label='reservoir', **kwargs)
discharge_down.plot(ax=ax[1], label='gauge', **kwargs)
# discharge_down.plot(ax=ax[1], label='gauge', **kwargs)
ax[1].legend(frameon=False)

ax[1].set(
    xlim=('2014-10-01', '2015-09-30'),
    ylabel='outflow (cms)'
);

***

### Subbasin Delineation

Subbasins are the intercatchments between gauging stations and dams, i.e., not the complete upstream catchment, but the subcatchment up to the upstream point (either station or dam).

In [ ]:
subbasins = delineate_subbasins(points.query("kind in ['station', 'dam']"), flwdir, uparea)
print(f'{len(subbasins)} subbasins')

# # export
# subbasins.to_file(cfg_camels.path_gis / 'subbasins.geojson', driver='GeoJSON')

### Basin Delineation

Basins are the whole river basin polygons. To get those with the same function (`delineate_subbasins`), we use only the `outlets`.

In [ ]:
basins = delineate_subbasins(
    points.xs('outlet', level='kind', drop_level=False), 
    flwdir, 
    uparea, 
    name='basin'
)
print(f'{len(basins)} basins')

# # export
# basins.to_file(cfg_camels.path_gis / 'basins.geojson', driver='GeoJSON')

### Plot

In [ ]:
# # plot stations, dams and subbasins
# size = 4
# points_kwgs = {
#     'station': dict(markersize=size),
#     'dam': dict(markersize=size, marker='v', c='k'),
#     'outlet': dict(markersize=size),
# }

# fig, ax = plt.subplots()
# for kind, kwgs in points_kwgs.items():
#     points.xs(kind, level='kind').plot(ax=ax, label=kind, zorder=2, **kwgs)
# subbasins.boundary.plot(ax=ax, edgecolor='dimgrey', lw=.2, zorder=0)
# basins.boundary.plot(ax=ax, edgecolor='k', lw=.5, zorder=1)
# ax.set_aspect('equal')
# ax.legend(loc=4, frameon=False)
# ax.axis('off');

### Rivers

In [ ]:
# Delineate streams
streams = gpd.GeoDataFrame.from_features(
    flwdir.streams(mask=uparea >= 1000),
    crs=crs
)

# keep only those within the basins
streams = gpd.sjoin(streams, basins, how='inner', predicate='intersects')

# # # export
# # streams.to_file(cfg_camels.path_gis / 'streams.geojson', driver='GeoJSON')

### Confluences

In [ ]:
# for ID in outlets.index:
#     print(ID)
#     parents = points[points.downstream_ID == ID]
#     if len(parents) < 2:
#         continue
#     else:
#         break

In [ ]:
# paths, dists = flwdir.path(parents['pixel'])
# n_parents = len(parents)
# confluences = []
# for i in range(n_parents):
#     for j in range(i + 1, n_parents):
#         print(i, '-', j)
#         for pixel in paths[i]:
#             if pixel in paths[j]:
#                 confluences.append(pixel.item())
#                 break
# confluences = list(set(confluences))

In [ ]:
# for ID2 in parents.index:
#     parents2 = points[points.downstream_ID == ID2]
#     if len(parents2) < 2:
#         continue
#     else:
#         print(ID2)
#         break

## Graph

### Create the graph

In [ ]:
# Initialize a Directed Graph
G = nx.DiGraph()

# Add nodes
# node_attrs = points[['kind', 'lat', 'lon', 'area', 'pixel', 'strahler']].to_dict('index')
node_attrs = points[['lat', 'lon', 'area', 'pixel', 'strahler']].to_dict('index')
for node_id, attrs in node_attrs.items():
    G.add_node(node_id, **attrs)
    
# Add edges
edges = points.dropna(subset=['downstream_ID'])
for ID, row in edges.iterrows():
    G.add_edge(u_of_edge=ID, v_of_edge=row['downstream_ID'], dist_m=row['dist_m'])

print(f"Graph created with {G.number_of_nodes()} stations and {G.number_of_edges()} connections.")

### Explore the graph

In [ ]:
# Find all nodes upstream of a target node
target_node = (1642, 'station')

upstream_stations = list(nx.ancestors(G, target_node))
print('ancestors:', upstream_stations)
source_node = upstream_stations[-1]

if G.has_node(source_node) and G.has_node(target_node):
    if nx.has_path(G, source_node, target_node):
        dist = nx.shortest_path_length(G, source_node, target_node, weight='dist_m')
        print(f"Distance {source_node}-{target_node}: {dist/1000:.2f} km")
    else:
        print("Nodes exist, but they are in different river basins!")
else:
    print("One of the station IDs is missing from the graph.")

In [ ]:
# identify outlet nodes
outlets = [node for node, degree in G.out_degree() if degree == 0]
print(f'{len(outlets)} outlet stations.')

# identify headwater nodes
headwaters = [node for node, degree in G.in_degree() if degree == 0]
print(f'{len(headwaters)} headwater stations.')

### Plot the graph

In [ ]:
plot_graph(
    graph=G,
    nodes=G.nodes,
    labels=False,
    streams=streams,
    basins=basins,
    title='Of Camels and Beavers',
    )